In [1]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)

from numcodecs import Blosc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
import zarr
import waveorder as wo
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor

In [2]:
def gather_data_paths(data_folder, sample_folders):
    dataPaths = []
    for j in sample_folders:
        temp = []
        temp_bg = []
        fold = os.path.join(data_folder,j)
        for dirs, folders, filenames in os.walk(fold):
            filenames.sort()

            for filename in filenames:

                ## Do not account for hidden files

                if "test" in filename:
                    continue

                if "Test" in filename:
                    continue

                if filename.endswith('.tif'):
                    temp.append(os.path.join(fold, os.path.join(dirs,filename)))

                else:
                    continue

        dataPaths.append(np.array(temp))
            
    data_paths_final = []
    for sample in dataPaths:
        temp_bg = []
        temp_C1 = []
        temp_C2 = []
        for i in sample:
            if 'old' in i:
                continue
            if 'BG' in i:
                temp_bg.append(i)
            if 'Coverslip_1' in i:
                temp_C1.append(i)
            if 'Coverslip_2'in i:
                temp_C2.append(i)

        data_paths_final.append([temp_bg,temp_C1,temp_C2])
        
    return data_paths_final

def init_zarr_store(data_path):
    chunk_size = (1, 65, 2048, 2448) # Dimension order PZYX
    compressor=Blosc(cname='zstd', clevel=1, shuffle=Blosc.SHUFFLE)
    
    data_shape = (5, 65, 2048, 2448)
    
    if not data_path.endswith('.zarr'):
        data_path += '.zarr'
    
    if not os.path.exists(data_path):
        os.makedirs(data_path)
        
    zarr_store = zarr.open(data_path, mode='a')
    
    
    datasets = ['Phase3D', 'Retardance', 'Orientation', 'BF']
    groups = ['C1', 'C2']
    
    for group in groups:
        zarr_store.create_group(group)
        for ds in datasets:
            zarr_store[group].zeros(ds, shape=data_shape, chunks=chunk_size, dtype='float32',
                                        compressor=compressor)
            
    return zarr_store

def gather_bg_and_positions(data_paths):
    positions_c1 = {}
    positions_c2 = {}
    
    with tiff.TiffFile(data_paths[1][0]) as tif:
        for idx, tiffpageseries in enumerate(tif.series):
            positions_c1[idx] = zarr.open(tiffpageseries.aszarr(), mode='r')   
            
    with tiff.TiffFile(data_paths[2][0]) as tif:
        for idx, tiffpageseries in enumerate(tif.series):
            positions_c2[idx] = zarr.open(tiffpageseries.aszarr(), mode='r') 
    
    bg_paths = data_paths[0]
    bg_paths.sort()
    bg_data = np.zeros([len(bg_paths),2048,2448])
    
    for i in range(len(bg_paths)):
        bg_data[i,:,:] = tiff.imread(bg_paths[i])

    return positions_c1, positions_c2, bg_data

## Gather Data Paths

In [3]:
data_folder = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549'
sample_folders = ['24hr_Mock',
                 '24hr_Mock_IFN',
                 '48hr_Mock_IFN',
                 '48hr_Mock',
                 '24hr_RSV',
                 '24hr_RSV_IFN',]

data_paths = gather_data_paths(data_folder, sample_folders)

In [4]:
data_paths[0]

[['/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/BG/State0.tif',
  '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/BG/State1.tif',
  '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/BG/State2.tif',
  '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/BG/State3.tif'],
 ['/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/Coverslip_1/_1/_1_MMStack.ome.tif',
  '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/Coverslip_1/_1/_1_MMStack_1.ome.tif',
  '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/Coverslip_1/_1/_1_MMStack_2.ome.tif',
  '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/Coverslip_1/_1/_1_MMStack_3.ome.tif',
  '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/24hr_Mock/Coverslip_1/_1/_1_MMStack_4.ome.tif'],
 ['/gpfs/CompMicro/rawdata

## Initialize Reconstruction

In [5]:
## Set Reconstruction Parameters
image_dim = (2048,2448)
wavelength = 525
swing = 0.05
N_channel = 4
NA_obj = 1.2
NA_illu = 0.4
mag = 40
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 3.45
bg_option = 'local_fit'
n_media = 1.33

save_dir = '/gpfs/CompMicro/projects/HAE/2021_02_03_40x_04NA_A549/'


reconstructor = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                                         pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)



Initializing Reconstructor...
Finished Initializing Reconstructor (1.41 min)
CPU times: user 1min 14s, sys: 10.2 s, total: 1min 24s
Wall time: 1min 24s


## Reconstruct Batch

In [ ]:
%%time
for sample in range(len(sample_folders[0])):
    c1, c2, bg = gather_bg_and_positions(data_paths[sample])
    
    zarr_store = init_zarr_store(save_dir+sample_folders[sample])
    
    for pos in range(len(c1)):
        
        ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c1[pos], bg, reconstructor, method = "Tikhonov",
                                                               reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                               lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
        
        zarr_store['C1']['BF'][pos] = BF_stack
        zarr_store['C1']['Retardance'][pos] = ret_stack
        zarr_store['C1']['Orientation'][pos] = ori_stack
        zarr_store['C1']['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))
        

    for pos2 in range(len(c2)):
        
        ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c2[pos2], bg, reconstructor, method = "Tikhonov",
                                                                       reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                                       lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
        zarr_store['C2']['BF'][pos2] = BF_stack
        zarr_store['C2']['Retardance'][pos2] = ret_stack
        zarr_store['C2']['Orientation'][pos2] = ori_stack
        zarr_store['C2']['Phase3D'][pos2] = np.transpose(phase3D,(2,0,1))
        

## Reconstruct Single Position

In [ ]:
%%time
sample = 1 #0-5
pos = 0   #0-4

c1, c2, bg = gather_bg_and_positions(data_paths[sample])

zarr_store = init_zarr_store(save_dir+'ReconOrderTest4')

ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c1[pos], bg, reconstructor, method = "Tikhonov",
                                                               reg_re = 1e-4, reg_im = 1e-4,rho = 1e-5, 
                                                               lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)

# zarr_store['C1']['BF'][pos] = BF_stack
# zarr_store['C1']['Retardance'][pos] = ret_stack
# zarr_store['C1']['Orientation'][pos] = ori_stack
zarr_store['C1']['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))

In [ ]:
## View Retardance
wo.visual.image_stack_viewer_fast(ret_stack, vrange=(0,20))

In [ ]:
## View Phase
wo.visual.image_stack_viewer_fast(np.transpose(phase3D,(2,0,1)))